In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker,
# to demonstrate that the attacker can be detected.

# BB84 with Attacker Overview:
# The protocol is identical to the plain version, but Eve intercepts the
# quantum channel between Alice and Bob.
#
# Attack strategy (intercept-resend):
#   Eve measures each qubit in a randomly chosen basis, then re-encodes and
#   forwards a new qubit to Bob based on her measurement result.
#   Because Eve does not know Alice's basis, she guesses wrong ~50% of the time.
#   When she guesses wrong, she disturbs the qubit state, introducing errors
#   that Alice and Bob can detect during error checking.
#
# Expected QBER with intercept-resend attack: ~25%
# Detection threshold used here: 15%
#
# Standard basis (basis=0):  |0> encodes bit 0,  |1> encodes bit 1
# Diagonal basis  (basis=1):  |+> encodes bit 0,  |-> encodes bit 1
#
# All random choices are generated by measuring a qubit in the
# |+> = 1/sqrt(2)(|0>+|1>) state � NOT Python's random module.

In [ ]:
# -----------------------------------------------------------------------------
# QUANTUM RANDOM NUMBER GENERATOR
# -----------------------------------------------------------------------------
# Applying a Hadamard gate to |0> produces 1/sqrt(2)(|0>+|1>).
# Measuring this superposition collapses it to 0 or 1 with equal probability,
# giving us a true quantum random bit.

backend = BasicSimulator()

def quantum_random_bit():
    """Return a single random bit (0 or 1) via quantum measurement."""
    qc = QuantumCircuit(1, 1)
    qc.h(0)           # Create superposition |+> = 1/sqrt(2)(|0>+|1>)
    qc.measure(0, 0)  # Collapse to 0 or 1 with equal probability
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])


def quantum_random_bits(n):
    """Return a list of n random bits generated via quantum measurement."""
    return [quantum_random_bit() for _ in range(n)]


# Sanity check
print("Sample random bits:", quantum_random_bits(10))

In [ ]:
# -----------------------------------------------------------------------------
# QUBIT ENCODING AND MEASUREMENT HELPERS
# -----------------------------------------------------------------------------
# BB84 uses two bases:
#   Standard basis (basis=0):  |0> encodes bit 0,  |1> encodes bit 1
#   Diagonal basis (basis=1):  |+> encodes bit 0,  |-> encodes bit 1
#
# Encoding:
#   Standard: start in |0>; apply X gate if bit=1  -> gives |0> or |1>
#   Diagonal: start in |0>; apply X if bit=1; then apply H -> gives |+> or |->
#
# Measurement:
#   Standard: measure directly in the Z basis (distinguishes |0> from |1>)
#   Diagonal: apply H first to rotate back to Z basis, then measure
#             (H|+> = |0>, H|-> = |1>)

def encode_qubit(bit, basis):
    """
    Encode a classical bit into a qubit using the specified basis.

    Parameters
    ----------
    bit   : int  -- 0 or 1, the value to encode
    basis : int  -- 0 for standard, 1 for diagonal

    Returns
    -------
    QuantumCircuit -- a 1-qubit circuit representing the encoded qubit
    """
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)   # Flip |0> to |1>
    if basis == 1:
        qc.h(0)   # Rotate to diagonal basis: |0>->|+>, |1>|->
    return qc


def measure_qubit(qubit_circuit, basis):
    """
    Measure a qubit circuit in the specified basis.

    Parameters
    ----------
    qubit_circuit : QuantumCircuit -- the encoded qubit (1-qubit, no classical bits)
    basis         : int            -- 0 for standard, 1 for diagonal

    Returns
    -------
    int -- the measured bit (0 or 1)
    """
    from qiskit import ClassicalRegister
    qc = qubit_circuit.copy()
    qc.add_register(ClassicalRegister(1))
    if basis == 1:
        qc.h(0)   # Rotate back from diagonal basis before measuring
    qc.measure(0, 0)
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

In [ ]:
# -----------------------------------------------------------------------------
# ALICE -- KEY PREPARATION
# -----------------------------------------------------------------------------
# Alice generates random bit values and random basis choices, then encodes
# each bit as a qubit ready to send over the quantum channel.

def alice_prepare(n):
    """
    Alice prepares n qubits for transmission.

    Parameters
    ----------
    n : int -- number of qubits to prepare

    Returns
    -------
    bits   : list[int]            -- Alice's random bit values
    bases  : list[int]            -- Alice's random basis choices (0=standard, 1=diagonal)
    qubits : list[QuantumCircuit] -- encoded qubits to send
    """
    bits  = quantum_random_bits(n)   # Random bit values
    bases = quantum_random_bits(n)   # Random basis choices
    qubits = [encode_qubit(b, basis) for b, basis in zip(bits, bases)]
    return bits, bases, qubits

In [ ]:
# -----------------------------------------------------------------------------
# CIRCUIT VISUALIZATIONS
# -----------------------------------------------------------------------------
# Show the four possible qubit encodings Alice can produce, so it is clear
# exactly which gates are applied for each (bit, basis) combination.
#
#   Standard basis, bit=0 : |0>  -- no gates needed, qubit starts in |0>
#   Standard basis, bit=1 : |1>  -- X gate flips |0> to |1>
#   Diagonal basis, bit=0 : |+>  -- H gate rotates |0> to |+>
#   Diagonal basis, bit=1 : |->  -- X then H rotates |0> to |->

print('=== Qubit Encoding Circuits ===')
labels = [
    ('Standard basis, bit=0  -->  |0>', 0, 0),
    ('Standard basis, bit=1  -->  |1>', 1, 0),
    ('Diagonal basis, bit=0  -->  |+>', 0, 1),
    ('Diagonal basis, bit=1  -->  |->', 1, 1),
]
for desc, bit, basis in labels:
    qc = encode_qubit(bit, basis)
    # Add a barrier to visually separate encoding from measurement
    qc.barrier()
    print(f'\n{desc}')
    print(qc.draw(output='text'))

In [ ]:
# -----------------------------------------------------------------------------
# EVE -- ATTACK STRATEGIES
# -----------------------------------------------------------------------------
# Two attack strategies are implemented:
#
# 1. FULL INTERCEPT-RESEND
#    Eve intercepts EVERY qubit, measures it in a random basis, re-encodes
#    her result, and forwards the new qubit to Bob.
#    Expected QBER: ~25%  (reliably detected above the 15% threshold)
#
# 2. PARTIAL INTERCEPT
#    Eve only intercepts ~50% of qubits at random; the rest pass undisturbed.
#    Expected QBER: ~12.5%  (sits just below the 15% threshold, harder to detect)
#    This shows that a stealthy attacker can stay under the radar with a
#    lower intercept rate, at the cost of learning less of the key.

def eve_intercept(qubits):
    """
    Eve intercepts and re-sends EVERY qubit (full intercept-resend attack).

    Parameters
    ----------
    qubits : list[QuantumCircuit] -- qubits sent by Alice

    Returns
    -------
    intercepted_qubits : list[QuantumCircuit] -- re-encoded qubits forwarded to Bob
    eve_bases          : list[int]            -- Eve's random basis choices
    eve_bits           : list[int]            -- Eve's measurement results
    """
    eve_bases = quantum_random_bits(len(qubits))  # Random basis for every qubit
    eve_bits  = []
    intercepted_qubits = []

    for qubit, basis in zip(qubits, eve_bases):
        # Measure the qubit -- this collapses the original quantum state
        measured_bit = measure_qubit(qubit, basis)
        eve_bits.append(measured_bit)
        # Re-encode and forward to Bob
        intercepted_qubits.append(encode_qubit(measured_bit, basis))

    return intercepted_qubits, eve_bases, eve_bits


def eve_partial_intercept(qubits):
    """
    Eve intercepts ~50% of qubits at random (partial intercept attack).

    Each qubit is independently intercepted or passed through based on a
    quantum random bit (0 = intercept, 1 = pass through).
    Qubits that are not intercepted reach Bob completely undisturbed.

    Expected QBER: ~12.5% -- just below the 15% detection threshold.

    Parameters
    ----------
    qubits : list[QuantumCircuit] -- qubits sent by Alice

    Returns
    -------
    forwarded_qubits : list[QuantumCircuit] -- qubits forwarded to Bob
    eve_bases        : list[int|None]       -- Eve's basis per qubit (None = not intercepted)
    eve_bits         : list[int|None]       -- Eve's result per qubit (None = not intercepted)
    intercept_mask   : list[bool]           -- True where Eve intercepted
    """
    forwarded_qubits = []
    eve_bases        = []
    eve_bits         = []
    intercept_mask   = []

    for qubit in qubits:
        # Use a quantum random bit to decide whether to intercept this qubit
        do_intercept = (quantum_random_bit() == 0)  # 50% chance
        intercept_mask.append(do_intercept)

        if do_intercept:
            # Eve intercepts: measure in a random basis and re-encode
            basis = quantum_random_bit()
            measured_bit = measure_qubit(qubit, basis)
            eve_bases.append(basis)
            eve_bits.append(measured_bit)
            forwarded_qubits.append(encode_qubit(measured_bit, basis))
        else:
            # Eve lets this qubit pass through completely undisturbed
            eve_bases.append(None)
            eve_bits.append(None)
            forwarded_qubits.append(qubit)

    n_intercepted = sum(intercept_mask)
    print(f'[Eve] Partial intercept: {n_intercepted}/{len(qubits)} qubits intercepted '
          f'({n_intercepted/len(qubits):.0%})')
    return forwarded_qubits, eve_bases, eve_bits, intercept_mask

In [ ]:
# -----------------------------------------------------------------------------
# EVE -- ADDITIONAL ATTACK STRATEGIES
# -----------------------------------------------------------------------------
#
# 3. BIASED INTERCEPT (Measure-in-One-Basis)
#    Eve always measures in the standard (Z) basis and re-encodes in that
#    same basis.  She never randomizes her basis choice.
#    When Alice used the standard basis, Eve gets the bit correctly.
#    When Alice used the diagonal basis, measurement collapses the state.
#    Expected QBER: ~25%  (same as full intercept-resend)
#
# 4. CNOT (ENTANGLEMENT-BASED) ATTACK
#    Eve entangles Alice's qubit with her own ancilla via a CNOT gate
#    (Alice = control, Eve = target), then measures only her ancilla.
#    For standard-basis qubits, CNOT produces |00> or |11> -- Eve learns
#    the bit without disturbing qubit 0.
#    For diagonal-basis qubits, CNOT creates a Bell state -- measuring
#    the ancilla collapses qubit 0 to |0> or |1>, destroying the encoding.
#    Expected QBER: ~25%  (the no-cloning theorem prevents silent copying)

def eve_biased_intercept(qubits):
    """
    Eve always measures in the STANDARD basis (biased attack).

    Unlike the random intercept-resend, Eve does not randomize her basis.
    She always uses basis=0 (standard/Z basis).

    Parameters
    ----------
    qubits : list[QuantumCircuit] -- qubits sent by Alice

    Returns
    -------
    intercepted_qubits : list[QuantumCircuit] -- re-encoded qubits
    eve_bits           : list[int]            -- Eve's measurement results
    """
    eve_bits = []
    intercepted_qubits = []

    for qubit in qubits:
        # Eve ALWAYS measures in the standard basis (basis=0)
        measured_bit = measure_qubit(qubit, 0)
        eve_bits.append(measured_bit)
        # Re-encode in the standard basis and forward to Bob
        intercepted_qubits.append(encode_qubit(measured_bit, 0))

    return intercepted_qubits, eve_bits


def eve_cnot_attack(qubits):
    """
    Eve entangles each qubit with her own ancilla via a CNOT gate.

    For each qubit:
      1. Eve prepares an ancilla |0>.
      2. Applies CNOT(Alice_qubit, ancilla).
      3. Measures her ancilla to extract information.
      4. Forwards the (now-disturbed) qubit to Bob.

    After CNOT + ancilla measurement:
      - Standard basis (|0>,|1>): CNOT gives |00> or |11>.
        Ancilla reveals Alice's bit; qubit 0 unchanged.
      - Diagonal basis (|+>,|->): CNOT creates Bell state.
        Ancilla measurement collapses qubit 0 to |0> or |1>.

    Parameters
    ----------
    qubits : list[QuantumCircuit] -- qubits sent by Alice

    Returns
    -------
    forwarded_qubits : list[QuantumCircuit] -- qubits forwarded to Bob
    eve_bits         : list[int]            -- Eve's ancilla results
    """
    eve_bits = []
    forwarded_qubits = []

    for qubit in qubits:
        # 2-qubit circuit: q0 = Alice's qubit, q1 = Eve's ancilla |0>
        qc = QuantumCircuit(2, 1)
        # Apply Alice's encoding gates onto qubit 0
        qc.compose(qubit, qubits=[0], inplace=True)
        # CNOT: Alice's qubit (control) -> Eve's ancilla (target)
        qc.cx(0, 1)
        # Eve measures her ancilla (qubit 1)
        qc.measure(1, 0)
        # Run the circuit
        compiled = transpile(qc, backend)
        result = backend.run(compiled, shots=1).result()
        counts = result.get_counts()
        eve_bit = int(list(counts.keys())[0])
        eve_bits.append(eve_bit)

        # After CNOT + measurement, qubit 0 is in |eve_bit> (standard basis)
        forwarded_qubits.append(encode_qubit(eve_bit, 0))

    return forwarded_qubits, eve_bits

In [ ]:
# -----------------------------------------------------------------------------
# BOB -- MEASUREMENT
# -----------------------------------------------------------------------------
# Bob receives the (possibly tampered) qubits from the channel and measures
# each one in a randomly chosen basis.

def bob_measure(qubits):
    """
    Bob measures each received qubit in a randomly chosen basis.

    Parameters
    ----------
    qubits : list[QuantumCircuit] -- qubits received from the channel

    Returns
    -------
    measured_bits : list[int] -- Bob's measurement outcomes
    bases         : list[int] -- Bob's random basis choices
    """
    bases = quantum_random_bits(len(qubits))   # Bob's random basis choices
    measured_bits = [measure_qubit(q, b) for q, b in zip(qubits, bases)]
    return measured_bits, bases

In [ ]:
# -----------------------------------------------------------------------------
# SIFTING -- BASIS RECONCILIATION
# -----------------------------------------------------------------------------
# Alice and Bob publicly compare their basis choices over a classical channel.
# They keep only the positions where both chose the same basis.
# The corresponding bits form the sifted key.

def sift_key(alice_bits, alice_bases, bob_bits, bob_bases):
    """
    Retain only the bits where Alice and Bob used the same basis.

    Parameters
    ----------
    alice_bits  : list[int] -- Alice's original bit values
    alice_bases : list[int] -- Alice's basis choices
    bob_bits    : list[int] -- Bob's measured bit values
    bob_bases   : list[int] -- Bob's basis choices

    Returns
    -------
    alice_key : list[int] -- Alice's sifted key
    bob_key   : list[int] -- Bob's sifted key
    """
    alice_key, bob_key = [], []
    for i in range(len(alice_bits)):
        if alice_bases[i] == bob_bases[i]:   # Bases match -- keep this bit
            alice_key.append(alice_bits[i])
            bob_key.append(bob_bits[i])
    return alice_key, bob_key

In [ ]:
# -----------------------------------------------------------------------------
# ERROR CHECKING
# -----------------------------------------------------------------------------
# Alice and Bob sacrifice a portion of their sifted key to estimate the
# Quantum Bit Error Rate (QBER).  A QBER above the threshold signals an attack.
#
# With an intercept-resend attack the expected QBER is ~25%, which is well
# above the 15% threshold used here.

DETECTION_THRESHOLD = 0.15   # Abort if more than 15% of check bits are wrong

def check_errors(alice_key, bob_key, sample_fraction=0.5):
    """
    Estimate the QBER by comparing a sample of the sifted key.

    Parameters
    ----------
    alice_key       : list[int] -- Alice's sifted key
    bob_key         : list[int] -- Bob's sifted key
    sample_fraction : float     -- fraction of the key used for error checking

    Returns
    -------
    error_rate      : float -- proportion of mismatched bits in the sample
    attack_detected : bool  -- True if error_rate exceeds DETECTION_THRESHOLD
    """
    n_sample = max(1, int(len(alice_key) * sample_fraction))
    errors = sum(a != b for a, b in zip(alice_key[:n_sample], bob_key[:n_sample]))
    error_rate = errors / n_sample
    attack_detected = error_rate > DETECTION_THRESHOLD
    return error_rate, attack_detected

In [ ]:
# -----------------------------------------------------------------------------
# PROTOCOL TABLE DISPLAY
# -----------------------------------------------------------------------------
# Prints a formatted table matching the lecture slide layout.
# Used twice in the attacker scenario:
#   1. Alice -> Eve  (what Eve sees on the quantum channel)
#   2. Alice -> Bob  (what Bob receives after Eve's interference)
#
# Qubit symbols: standard basis -> 0 / 1 ; diagonal basis -> + / -
# Columns marked * are positions where both parties' bases matched.
# The receiver's bit shows ? where bases did not match.

def qubit_symbol(bit, basis):
    """Return the qubit state symbol for a given bit and basis."""
    if basis == 0:          # Standard basis: |0> or |1>
        return str(bit)
    else:                   # Diagonal basis: |+> or |->
        return '+' if bit == 0 else '-'


def print_bb84_table(title, alice_bits, alice_bases,
                     receiver_bits, receiver_bases, receiver_label='B'):
    """
    Print a BB84 protocol table to the console.

    Parameters
    ----------
    title          : str       -- heading printed above the table
    alice_bits     : list[int] -- Alice's raw bit values
    alice_bases    : list[int] -- Alice's basis choices (0=standard, 1=diagonal)
    receiver_bits  : list[int] -- receiver's measured bit values
    receiver_bases : list[int] -- receiver's basis choices
    receiver_label : str       -- single-character label for the receiver row (default 'B')
    """
    n   = len(alice_bits)
    col = 3   # fixed column width

    # Positions where Alice's and receiver's bases match
    match = [alice_bases[i] == receiver_bases[i] for i in range(n)]

    idx_row   = [str(i)                                         for i in range(n)]
    match_row = ['*' if match[i] else ' '                       for i in range(n)]
    a_bit_row = [str(alice_bits[i])                             for i in range(n)]
    a_bas_row = ['s' if alice_bases[i] == 0 else 'd'            for i in range(n)]
    qubit_row = [qubit_symbol(alice_bits[i], alice_bases[i])    for i in range(n)]
    r_bas_row = ['s' if receiver_bases[i] == 0 else 'd'         for i in range(n)]
    r_bit_row = [str(receiver_bits[i]) if match[i] else '?'     for i in range(n)]

    def fmt_row(label, cells):
        return f'{label:<10}' + ''.join(c.center(col) for c in cells)

    sep = '-' * (10 + col * n)
    print(f'\n{title}')
    print(sep)
    print(fmt_row('Index',              idx_row))
    print(fmt_row('Match',              match_row))   # * = bases matched
    print(sep)
    print(fmt_row('A bit',              a_bit_row))
    print(fmt_row('A basis',            a_bas_row))   # s=standard, d=diagonal
    print(fmt_row('Qubit',              qubit_row))   # 0/1 or +/-
    print(sep)
    print(fmt_row(f'{receiver_label} basis', r_bas_row))
    print(fmt_row(f'{receiver_label} bit',   r_bit_row))  # ? where bases differ
    print(sep)


# -----------------------------------------------------------------------------
# FULL BB84 PROTOCOL -- WITH ATTACKER (INTERCEPT-RESEND)
# -----------------------------------------------------------------------------
# Runs the complete BB84 protocol with Eve performing an intercept-resend attack.

def run_bb84_attacker(n_qubits=20):
    """
    Simulate the BB84 protocol with an intercept-resend attacker (Eve).

    Parameters
    ----------
    n_qubits : int -- number of qubits Alice sends

    Returns
    -------
    final_key : list[int] -- shared key if no attack detected, else empty list
    """
    print(f'=== BB84 Protocol (Intercept-Resend Attack) -- {n_qubits} qubits ===')

    # -- ALICE ----------------------------------------------------------------
    alice_bits, alice_bases, qubits = alice_prepare(n_qubits)

    # -- EVE (quantum channel) ------------------------------------------------
    # Eve intercepts every qubit, measures it in a random basis, and re-sends.
    intercepted, eve_bases, eve_bits = eve_intercept(qubits)

    # Table 1: Alice vs Eve -- shows what Eve intercepted
    # * marks positions where Eve guessed Alice's basis correctly (no disturbance)
    print_bb84_table(
        'TABLE 1 -- Alice -> Eve (what Eve intercepted)',
        alice_bits, alice_bases,
        eve_bits,   eve_bases,
        receiver_label='E'
    )

    # -- BOB ------------------------------------------------------------------
    # Bob receives Eve's re-sent qubits (not Alice's originals)
    bob_bits, bob_bases = bob_measure(intercepted)

    # Table 2: Alice vs Bob -- shows the errors Eve introduced
    # * marks positions where Alice and Bob's bases matched (sifted positions)
    # Errors in the sifted key are visible where A bit != B bit on * columns
    print_bb84_table(
        'TABLE 2 -- Alice -> Bob (after Eve\'s interference)',
        alice_bits, alice_bases,
        bob_bits,   bob_bases,
        receiver_label='B'
    )

    # -- SIFTING (public classical channel) -----------------------------------
    alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    print(f'\n[Sifting] Alice sifted key : {alice_key}')
    print(f'[Sifting] Bob   sifted key : {bob_key}')

    # -- ERROR CHECKING -------------------------------------------------------
    error_rate, attack_detected = check_errors(alice_key, bob_key)
    print(f'\n[Error Check] QBER = {error_rate:.2%}  (threshold = {DETECTION_THRESHOLD:.0%})')

    if attack_detected:
        print('[Error Check] Attack DETECTED -- aborting protocol.')
        return []
    else:
        # Unlikely with a full intercept-resend attack, but possible with
        # small qubit counts due to statistical fluctuation.
        print('[Error Check] No attack detected (channel appears secure).')

    # -- FINAL KEY ------------------------------------------------------------
    n_check = max(1, int(len(alice_key) * 0.5))
    final_key = alice_key[n_check:]
    print(f'\n[Key] Final shared key ({len(final_key)} bits): {final_key}')
    return final_key


# Run the protocol with the attacker
key = run_bb84_attacker(n_qubits=20)

In [ ]:
# -----------------------------------------------------------------------------
# FULL BB84 PROTOCOL -- WITH PARTIAL INTERCEPT ATTACK
# -----------------------------------------------------------------------------
# Eve only intercepts ~50% of qubits.  The expected QBER is ~12.5%, which
# sits just below the 15% detection threshold -- demonstrating that a
# stealthy attacker can sometimes evade detection at the cost of learning
# only a fraction of the key.

def run_bb84_partial(n_qubits=20):
    """
    Simulate the BB84 protocol with a partial intercept attack (Eve ~50%).

    Parameters
    ----------
    n_qubits : int -- number of qubits Alice sends

    Returns
    -------
    final_key : list[int] -- shared key if no attack detected, else empty list
    """
    print(f'=== BB84 Protocol (Partial Intercept Attack) -- {n_qubits} qubits ===')

    # -- ALICE ----------------------------------------------------------------
    alice_bits, alice_bases, qubits = alice_prepare(n_qubits)

    # -- EVE (quantum channel) ------------------------------------------------
    # Eve intercepts ~50% of qubits; the rest pass through undisturbed.
    forwarded, eve_bases, eve_bits, mask = eve_partial_intercept(qubits)

    # Table 1: Alice vs Eve -- only intercepted positions are shown with Eve's result
    # Non-intercepted positions show None in eve_bases; we substitute 's' for display
    eve_bases_display = [b if b is not None else alice_bases[i]
                         for i, b in enumerate(eve_bases)]
    eve_bits_display  = [b if b is not None else alice_bits[i]
                         for i, b in enumerate(eve_bits)]
    print_bb84_table(
        'TABLE 1 -- Alice -> Eve (intercepted positions only; others pass through)',
        alice_bits, alice_bases,
        eve_bits_display, eve_bases_display,
        receiver_label='E'
    )

    # -- BOB ------------------------------------------------------------------
    bob_bits, bob_bases = bob_measure(forwarded)

    # Table 2: Alice vs Bob -- errors only appear at intercepted positions
    print_bb84_table(
        'TABLE 2 -- Alice -> Bob (errors only at intercepted positions)',
        alice_bits, alice_bases,
        bob_bits, bob_bases,
        receiver_label='B'
    )

    # -- SIFTING --------------------------------------------------------------
    alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    print(f'\n[Sifting] Alice sifted key : {alice_key}')
    print(f'[Sifting] Bob   sifted key : {bob_key}')

    # -- ERROR CHECKING -------------------------------------------------------
    error_rate, attack_detected = check_errors(alice_key, bob_key)
    print(f'\n[Error Check] QBER = {error_rate:.2%}  (threshold = {DETECTION_THRESHOLD:.0%})')

    if attack_detected:
        print('[Error Check] Attack DETECTED -- aborting protocol.')
        return []
    else:
        print('[Error Check] Attack NOT detected -- Eve stayed under the threshold.')

    # -- FINAL KEY ------------------------------------------------------------
    n_check = max(1, int(len(alice_key) * 0.5))
    final_key = alice_key[n_check:]
    print(f'\n[Key] Final shared key ({len(final_key)} bits): {final_key}')
    return final_key


# Run the partial intercept scenario
key_partial = run_bb84_partial(n_qubits=20)

In [ ]:
# -----------------------------------------------------------------------------
# FULL BB84 PROTOCOL -- WITH BIASED ATTACK (Standard Basis Only)
# -----------------------------------------------------------------------------
# Eve always measures in the standard basis.  Expected QBER: ~25%.
# Demonstrates that even a deterministic strategy cannot avoid detection.

def run_bb84_biased(n_qubits=20):
    """
    Simulate BB84 with a biased attacker (Eve uses standard basis only).
    """
    print(f'=== BB84 (Biased Attack -- Standard Basis Only) -- {n_qubits} qubits ===')

    # -- ALICE ----------------------------------------------------------------
    alice_bits, alice_bases, qubits = alice_prepare(n_qubits)

    # -- EVE ------------------------------------------------------------------
    intercepted, eve_bits = eve_biased_intercept(qubits)
    eve_bases = [0] * n_qubits   # Eve always used standard basis

    print_bb84_table(
        'TABLE 1 -- Alice -> Eve (Eve always uses standard basis)',
        alice_bits, alice_bases,
        eve_bits, eve_bases,
        receiver_label='E'
    )

    # -- BOB ------------------------------------------------------------------
    bob_bits, bob_bases = bob_measure(intercepted)

    print_bb84_table(
        'TABLE 2 -- Alice -> Bob (after Eve\'s biased interference)',
        alice_bits, alice_bases,
        bob_bits, bob_bases,
        receiver_label='B'
    )

    # -- SIFTING --------------------------------------------------------------
    alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    print(f'\n[Sifting] Alice sifted key : {alice_key}')
    print(f'[Sifting] Bob   sifted key : {bob_key}')

    # -- ERROR CHECKING -------------------------------------------------------
    error_rate, attack_detected = check_errors(alice_key, bob_key)
    print(f'\n[Error Check] QBER = {error_rate:.2%}  (threshold = {DETECTION_THRESHOLD:.0%})')

    if attack_detected:
        print('[Error Check] Attack DETECTED -- aborting protocol.')
        return []
    else:
        print('[Error Check] No attack detected (channel appears secure).')

    n_check = max(1, int(len(alice_key) * 0.5))
    final_key = alice_key[n_check:]
    print(f'\n[Key] Final shared key ({len(final_key)} bits): {final_key}')
    return final_key


# Run the biased attack scenario
key_biased = run_bb84_biased(n_qubits=20)

In [ ]:
# -----------------------------------------------------------------------------
# FULL BB84 PROTOCOL -- WITH CNOT (ENTANGLEMENT) ATTACK
# -----------------------------------------------------------------------------
# Eve uses a CNOT gate to entangle Alice's qubit with her own ancilla,
# then measures the ancilla.  This is a more sophisticated quantum attack
# that demonstrates the no-cloning theorem: even entanglement-based
# strategies cannot silently copy quantum information.
# Expected QBER: ~25%.

def run_bb84_cnot(n_qubits=20):
    """
    Simulate BB84 with an entanglement-based (CNOT) attacker.
    """
    print(f'=== BB84 (CNOT / Entanglement Attack) -- {n_qubits} qubits ===')

    # -- ALICE ----------------------------------------------------------------
    alice_bits, alice_bases, qubits = alice_prepare(n_qubits)

    # -- EVE (CNOT attack) ----------------------------------------------------
    forwarded, eve_bits = eve_cnot_attack(qubits)
    # Eve has no explicit basis -- she uses CNOT, not basis measurement
    # For display purposes, treat Eve's effective basis as standard (0)
    eve_bases = [0] * n_qubits

    print_bb84_table(
        'TABLE 1 -- Alice -> Eve (Eve uses CNOT entanglement)',
        alice_bits, alice_bases,
        eve_bits, eve_bases,
        receiver_label='E'
    )

    # -- BOB ------------------------------------------------------------------
    bob_bits, bob_bases = bob_measure(forwarded)

    print_bb84_table(
        'TABLE 2 -- Alice -> Bob (after Eve\'s CNOT interference)',
        alice_bits, alice_bases,
        bob_bits, bob_bases,
        receiver_label='B'
    )

    # -- SIFTING --------------------------------------------------------------
    alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    print(f'\n[Sifting] Alice sifted key : {alice_key}')
    print(f'[Sifting] Bob   sifted key : {bob_key}')

    # -- ERROR CHECKING -------------------------------------------------------
    error_rate, attack_detected = check_errors(alice_key, bob_key)
    print(f'\n[Error Check] QBER = {error_rate:.2%}  (threshold = {DETECTION_THRESHOLD:.0%})')

    if attack_detected:
        print('[Error Check] Attack DETECTED -- aborting protocol.')
        return []
    else:
        print('[Error Check] No attack detected (channel appears secure).')

    n_check = max(1, int(len(alice_key) * 0.5))
    final_key = alice_key[n_check:]
    print(f'\n[Key] Final shared key ({len(final_key)} bits): {final_key}')
    return final_key


# Run the CNOT attack scenario
key_cnot = run_bb84_cnot(n_qubits=20)

In [ ]:
# -----------------------------------------------------------------------------
# VERIFICATION -- COMPARING ALL ATTACK TYPES OVER MULTIPLE TRIALS
# -----------------------------------------------------------------------------
# Run 10 trials of each attack and compare detection rates and average QBERs.
#
# Full intercept:    QBER ~25% -- should be detected reliably
# Partial intercept: QBER ~12.5% -- may slip under the 15% threshold
# Biased intercept:  QBER ~25% -- should be detected reliably
# CNOT attack:       QBER ~25% -- should be detected reliably

N_TRIALS  = 10
N_QUBITS  = 30

print(f'Running {N_TRIALS} trials of each attack ({N_QUBITS} qubits per trial)...\n')

# Helper to run a batch of trials for a given attack
def run_trials(label, attack_fn):
    """
    Run N_TRIALS of the BB84 protocol with the given attack function.

    Parameters
    ----------
    label     : str      -- display name for the attack
    attack_fn : callable -- function(qubits) -> (forwarded_qubits, ...)
                            Must return forwarded qubits as first element.
    """
    print(f'--- {label} ---')
    detected = 0
    qbers = []
    for trial in range(1, N_TRIALS + 1):
        a_bits, a_bases, qubits = alice_prepare(N_QUBITS)
        result = attack_fn(qubits)
        forwarded = result[0]   # first return value is always the forwarded qubits
        b_bits, b_bases = bob_measure(forwarded)
        a_key, b_key = sift_key(a_bits, a_bases, b_bits, b_bases)
        rate, det = check_errors(a_key, b_key)
        detected += int(det)
        qbers.append(rate)
        status = 'DETECTED' if det else 'missed'
        print(f'  Trial {trial:2d}: sifted={len(a_key):2d}  QBER={rate:.2%}  --> {status}')
    avg = sum(qbers) / len(qbers)
    print(f'\n  Detected: {detected}/{N_TRIALS}  |  Avg QBER: {avg:.2%}\n')
    return detected, avg

# Run all four attack types
d1, a1 = run_trials('1. Full Intercept-Resend',    eve_intercept)
d2, a2 = run_trials('2. Partial Intercept (~50%)',  eve_partial_intercept)
d3, a3 = run_trials('3. Biased (Standard Only)',    eve_biased_intercept)
d4, a4 = run_trials('4. CNOT (Entanglement)',       eve_cnot_attack)

# Summary table
print('=' * 65)
print(f'{"Attack":<30} {"Detected":>10} {"Avg QBER":>12}')
print('-' * 65)
print(f'{"Full intercept-resend":<30} {d1:>5}/{N_TRIALS:<4} {a1:>11.2%}')
print(f'{"Partial intercept (~50%)":<30} {d2:>5}/{N_TRIALS:<4} {a2:>11.2%}')
print(f'{"Biased (standard only)":<30} {d3:>5}/{N_TRIALS:<4} {a3:>11.2%}')
print(f'{"CNOT (entanglement)":<30} {d4:>5}/{N_TRIALS:<4} {a4:>11.2%}')
print('=' * 65)
print('\nFull/biased/CNOT attacks produce ~25% QBER and are reliably detected.')
print('Partial intercept produces ~12.5% QBER and may evade the 15% threshold.')